<a href="https://colab.research.google.com/github/YASH-HASH-PIXEL/-ANN-Linear_Regression/blob/main/PROJECT%20part-02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from transformers import pipeline, AutoTokenizer, GenerationConfig
import gradio as gr

class ChatBot:
    def __init__(self, pipe, history=None):
        self.pipe = pipe
        self.history = history or []
        self.tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
        self.tokenizer.pad_token = self.tokenizer.eos_token

        self.gen_config = GenerationConfig(
            max_new_tokens=150,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.7,
            pad_token_id=self.tokenizer.eos_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
            max_length=None,
        )

    def get_response(self, user_input):
        # Keep only the last 4 turns
        context_parts = [turn["text"] for turn in self.history[-4:]]
        context_parts.append(user_input)
        context_str = "<|endoftext|>".join(context_parts) + "<|endoftext|>"

        outputs = self.pipe(
            context_str,
            generation_config=self.gen_config,
            return_full_text=False,
        )
        reply = outputs[0]["generated_text"].strip()

        if "\n" in reply:
            reply = reply.split("\n")[0]
        if not reply:
            reply = "I'm not sure what to say."

        self.history.append({"role": "user", "text": user_input})
        self.history.append({"role": "bot", "text": reply})
        return reply

    def reset(self):
        self.history = []
        print("[Conversation history cleared]", flush=True)


# Gradio UI integration

def get_chatbot_response(message, history, bot_state):
    """Called every time the user sends a message."""
    if bot_state is None:
        # First call in this session – create a fresh ChatBot instance
        bot_state = ChatBot(pipe)

    response = bot_state.get_response(message)
    # history is a list of [user, bot] pairs – Gradio handles the display
    history.append([message, response])
    # Return: new message input (empty), updated history, unchanged bot state
    return "", history, bot_state

def reset_conversation(bot_state):
    """Reset the bot's history and clear the UI chat log."""
    if bot_state is not None:
        bot_state.reset()
    return [], None   # clear UI history and reset bot_state (will be recreated)

# Load the model once (shared across all sessions)
print("Loading model... (this may take a few minutes the first time)")
pipe = pipeline(
    "text-generation",
    model="microsoft/DialoGPT-medium",
    device=-1          # -1 for CPU, use 0 if you have a GPU
)
print("Model loaded successfully.")

# Build the Gradio interface
with gr.Blocks(title="DialoGPT Chatbot") as demo:
    gr.Markdown("# 🤖 DialoGPT Chatbot")
    gr.Markdown("A conversational AI based on Microsoft's DialoGPT-medium. Type a message and press Enter.")

    # Stateful components
    bot_state = gr.State(None)          # holds the ChatBot instance for this session
    chatbot = gr.Chatbot(label="Conversation")
    msg = gr.Textbox(label="Your message", placeholder="Say something...")
    clear = gr.Button("Clear conversation")

    # Wire up the events
    msg.submit(get_chatbot_response, [msg, chatbot, bot_state], [msg, chatbot, bot_state])
    clear.click(reset_conversation, [bot_state], [chatbot, bot_state])

if __name__ == "__main__":
    demo.launch(server_name="0.0.0.0", server_port=7860)

Loading model... (this may take a few minutes the first time)


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully.


/tmp/ipykernel_10378/1258148516.py:85: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Conversation")
/tmp/ipykernel_10378/1258148516.py:85: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Conversation")


OSError: Cannot find empty port in range: 7860-7860. You can specify a different port by setting the GRADIO_SERVER_PORT environment variable or passing the `server_port` parameter to `launch()`.

In [ ]:
from transformers import pipeline, AutoTokenizer, GenerationConfig

class ChatBot:
    def __init__(self, pipe, history=None):
        self.pipe = pipe
        self.history = history or []
        self.tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
        self.tokenizer.pad_token = self.tokenizer.eos_token

        self.gen_config = GenerationConfig(
            max_new_tokens=150,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.7,
            pad_token_id=self.tokenizer.eos_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
            max_length=None,
        )

    def get_response(self, user_input):
        # Keep only the last 4 turns
        context_parts = [turn["text"] for turn in self.history[-4:]]
        context_parts.append(user_input)
        context_str = "<|endoftext|>".join(context_parts) + "<|endoftext|>"

        outputs = self.pipe(
            context_str,
            generation_config=self.gen_config,
            return_full_text=False,
        )
        reply = outputs[0]["generated_text"].strip()

        if "\n" in reply:
            reply = reply.split("\n")[0]
        if not reply:
            reply = "I'm not sure what to say."

        self.history.append({"role": "user", "text": user_input})
        self.history.append({"role": "bot", "text": reply})
        return reply

    def reset(self):
        self.history = []
        print("[Conversation history cleared]", flush=True)


if __name__ == "__main__":
    # Create the text-generation pipeline
    # On first run, this downloads the model (~400 MB) – be patient
    print("Loading model... (this may take a few minutes the first time)")
    pipe = pipeline(
        "text-generation",
        model="microsoft/DialoGPT-medium",
        device=-1   # -1 for CPU, use 0 for GPU if available
    )
    print("Model loaded successfully.")

    # Instantiate the chatbot
    bot = ChatBot(pipe)

    # Have a short conversation
    print("User: Hello, how are you?")
    response = bot.get_response("Hello, how are you?")
    print("Bot:", response)

    print("User: What's your favorite color?")
    response = bot.get_response("What's your favorite color?")
    print("Bot:", response)

    print("User: Tell me a joke.")
    response = bot.get_response("Tell me a joke.")
    print("Bot:", response)

Loading model... (this may take a few minutes the first time)


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully.
User: Hello, how are you?
Bot: I'm good, how are you?
User: What's your favorite color?
Bot: Orange. My favorite color is orange, I also love orange.
User: Tell me a joke.
Bot: Are you orange or are you not orange?
